# 📘 SafeDeck Detection V6: YOLOv11 학습 및 배포 파이프라인

> **현재 우선순위**: 모델 학습 및 성능 개선에 집중하며, TensorRT 변환 및 배포는 최종 모델 확정 후 진행 예정입니다.

---

## 🗺️ 노트북 구조 가이드 (반드시 읽어주세요!)

이 노트북은 **Phase별로 구성**되어 있으며, **실행 환경에 따라 다른 Phase를 실행**해야 합니다.

```
┌─────────────────────────────────────────────────────────────────┐
│                     노트북 Phase 구조                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  📍 Phase 1: YOLOv11 학습 + MLflow 실험 추적                    │
│     └─ 환경: Windows (개발 환경)                                │
│     └─ 실행: ✅ 필수                                            │
│     └─ 결과: best.pt 모델 생성                                  │
│                                                                 │
│  ─────────────────────────────────────────────────────────────  │
│                                                                 │
│  📍 Phase 2-A: 모델 변환 - ONNX (Windows)                      │
│     └─ 환경: Windows (개발 환경)                                │
│     └─ 실행: ✅ 권장 (범용 포맷, 검증용)                        │
│     └─ 결과: best.onnx 생성                                     │
│     └─ 역할: 디버깅, 백업, 플랫폼 호환성                        │
│                                                                 │
│  ─────────────────────────────────────────────────────────────  │
│                                                                 │
│  📍 Phase 2-B: 모델 변환 - TensorRT (Jetson)                   │
│     └─ 환경: Jetson Orin Nano ⚠️ (임베디드 배포 환경)          │
│     └─ 실행: ⚠️ Jetson에서만 실행!                             │
│     └─ 결과: best.engine 생성                                   │
│     └─ 역할: GPU 최적화, 실시간 추론                            │
│                                                                 │
│  ─────────────────────────────────────────────────────────────  │
│                                                                 │
│  📍 Phase 3: 속도 벤치마크 (PyTorch vs TensorRT)               │
│     └─ 환경: Jetson Orin Nano (배포 환경)                      │
│     └─ 실행: ⏭️ 선택 (성능 측정용)                             │
│     └─ 결과: FPS, 지연시간 비교                                 │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### ⚠️ 중요: 환경별 실행 가이드

| Phase | Windows | Jetson | 필수 여부 | 역할 |
|-------|---------|--------|----------|------|
| **Phase 1** | ✅ 실행 | - | 필수 | 모델 학습 |
| **Phase 2-A** | ✅ 실행 | - | 권장 | ONNX 변환 (범용) |
| **Phase 2-B** | ❌ 건너뜀 | ✅ 실행 | 필수 | TensorRT 변환 |
| **Phase 3** | - | ✅ 실행 | 선택 | 성능 벤치마크 |

### 🎯 빠른 시작 (Quick Start)

**Windows 개발자:**
1. Phase 1 실행 → 모델 학습
2. Phase 2-A 실행 → ONNX 변환
3. Phase 2-B는 **건너뛰기** (Jetson 전용)

**Jetson 배포자:**
1. Windows에서 학습된 `best.pt` 파일을 Jetson으로 복사
2. Phase 2-B 실행 → TensorRT 변환
3. Phase 3 실행 → 성능 측정

### 💡 왜 Phase가 나뉘나요?

**Phase 2를 A/B로 나눈 이유:**
- **Phase 2-A (Windows)**: TensorRT가 Windows에서 설치 불가 → ONNX로 대체
- **Phase 2-B (Jetson)**: TensorRT는 Jetson에 기본 설치됨 → GPU 최적화

**결론:** Windows에서는 Phase 2-B 셀을 **실행하지 마세요!**

---

## V6 학습 결과 (2026-01-21) - 목표 달성!

### 핵심 성과

| 지표 | V5 (이전) | V6 (현재) | 변화 | 목표 달성 |
|------|-----------|----------|------|----------|
| **mAP50** | 0.6995 | **0.7972** | **+9.8%p** | ✅ (목표: 0.74+) |
| **mAP50-95** | 0.5033 | **0.5740** | **+7.1%p** | ✅ |
| **Scratch mAP50** | 0.116 | **0.812** | **+696% (7배)** | ✅ |
| **학습 에포크** | 49/100 | 43/100 | -6 에포크 | 더 빠른 수렴 |
| **Best 에포크** | 44 | 38 | - | - |
| **종료 상태** | STUCK | NORMAL | 정상 완료 | ✅ |

---

## 클래스별 성능 비교: V5 vs V6

| 클래스 | V5 mAP50 | V6 mAP50 | 변화 | 비고 |
|--------|----------|----------|------|------|
| **Scratch** | **0.116** | **0.812** | **+696%** | 데이터 증강 효과! |
| Blistering | 0.983 | 0.976 | -0.7% | 유지 |
| WeldingDamage | 0.970 | 0.967 | -0.3% | 유지 |
| Corrosion | 0.968 | 0.965 | -0.3% | 유지 |
| Crack | 0.853 | 0.871 | +1.8% | 개선 |
| Pinhole | 0.556 | 0.545 | -1.1% | 유지 |
| Peeling | 0.451 | 0.447 | -0.4% | 유지 |

**결론**: Scratch 클래스 성능이 7배 향상되면서 다른 클래스 성능은 유지됨

---

## Scratch 데이터셋 비교: V5 vs V6

| 항목 | V5 (이전) | V6 (현재) | 변화 |
|------|-----------|----------|------|
| **모델** | yolo11n | yolo11n | - |
| **Scratch 데이터 (Train)** | 9장 | **1,637장** | **+1,628장 (182배)** |
| **Scratch 데이터 (Val)** | 2장 | **410장** | **+408장 (205배)** |
| **총 Train 이미지** | ~10,000장 | **~11,637장** | +1,637장 |
| **총 Val 이미지** | ~2,500장 | **~2,910장** | +410장 |

---

## V6 데이터 소스

- **출처**: AI Hub "선박 도장 품질 측정 데이터" (데이터셋 ID: 71447)
- **다운로드 항목**: Validation 스크래치 이미지 + 라벨 (4.5GB)
- **변환**: COCO JSON → YOLO TXT 형식 (`convert_aihub_to_yolo.py`)
- **분배**: Train 80% (1,637장) / Val 20% (410장)
- **파일명**: `aihub_scratch_` prefix 추가

---

## V6 실험 결론

### 성공 요인
1. **데이터 불균형 해소**: Scratch 클래스 9장 → 1,637장 (182배 증가)
2. **학습 안정화**: STUCK → NORMAL_COMPLETION으로 개선
3. **전체 성능 향상**: mAP50 0.6995 → 0.7972 (+9.8%p)

### 남은 개선 과제
| 클래스 | 현재 mAP50 | 상태 |
|--------|-----------|------|
| Pinhole | 0.545 | 개선 필요 |
| Peeling | 0.447 | 개선 필요 |

### 다음 단계
- [x] V6 학습 완료 및 목표 달성
- [ ] Phase 2-A: ONNX 변환 (Windows)
- [ ] Phase 2-B: TensorRT 변환 (Jetson)
- [ ] Phase 3: 속도 벤치마크
- [ ] (선택) Pinhole/Peeling 데이터 추가 증강

---

# SafeDeck Detection V5 결과 (이전 학습) - V6에서 개선됨

> **V6 업데이트**: Scratch 데이터 증강으로 mAP50 0.6995 → **0.7972** 달성! (아래 V5 결과는 참고용)

---

## V5 학습 결과 요약 (2026-01-20)

### 성능 지표

| 지표 | V5 결과 | V6 결과 | 개선 |
|------|---------|---------|------|
| **mAP50** | 0.6995 | **0.7972** | **+9.8%p** |
| **mAP50-95** | 0.5033 | **0.5740** | **+7.1%p** |
| **Precision** | 0.7723 | 0.7520 | -2.0%p |
| **Recall** | 0.6803 | **0.7660** | **+8.6%p** |
| **학습 에포크** | 49/100 | 43/100 | 조기 종료 |
| **Best 에포크** | 44 | 38 | - |

### V5 클래스별 성능 (V6 비교)

| 클래스 | V5 mAP50 | V6 mAP50 | 변화 |
|--------|----------|----------|------|
| Blistering | 0.983 | 0.976 | -0.7% |
| WeldingDamage | 0.970 | 0.967 | -0.3% |
| Corrosion | 0.968 | 0.965 | -0.3% |
| Crack | 0.853 | 0.871 | +1.8% |
| Pinhole | 0.556 | 0.545 | -1.1% |
| Peeling | 0.451 | 0.447 | -0.4% |
| **Scratch** | **0.116** | **0.812** | **+696%** |

### V5 문제점 → V6 해결

| 문제 | V5 상태 | V6 해결 |
|------|---------|---------|
| Scratch 데이터 부족 | 9장 | 1,637장 추가 |
| Scratch mAP 저조 | 0.116 | **0.812** |
| 전체 mAP 목표 미달 | 0.6995 | **0.7972** |
| 학습 정체 (STUCK) | 발생 | 정상 완료 |

---

## MLOps 적용 성과 ✅

### 1. 실험 추적 (Experiment Tracking)
- **파라미터 로깅**: epochs, batch, lr0, optimizer, patience 자동 기록
- **시스템 정보**: GPU (RTX 4070), CUDA, PyTorch 버전 기록
- **메트릭 로깅**: train_loss, val_loss, mAP50 에포크별 기록

### 2. 자동화된 학습 제어
- **Smart Training Monitor**: Train/Val Loss 실시간 분석
- **자동 조기 종료**: STUCK 감지 → V5: 49 에포크, V6: 43 에포크에서 중단
- **시간 절약**: 100 에포크 → 43~49 에포크 (50+ 에포크 절약)

### 3. 모델 버전 관리
- **아티팩트 저장**: best.pt MLflow에 자동 저장
- **실험 비교**: MLflow UI에서 V5, V6 실험 비교 가능

---

## 전체 파이프라인
```
YOLOv11 학습 (PyTorch) → MLflow 실험 추적 → TensorRT 변환 → Jetson 배포 최적화
```

## 최종 성과
- **정확도**: mAP50 **0.7972** 달성 (목표 0.74+ 초과 달성)
- **속도**: TensorRT로 추론 속도 2~3배 향상 (예정)
- **배포**: Jetson Orin Nano 최적화 (예정)
- **MLOps**: MLflow 기반 실험 관리 ✅ 완료

In [31]:
!pip install mlflow

In [2]:
import os
import mlflow
from datetime import datetime

# 경로 설정
base_path = r'C:\Users\SSAFY\Desktop\S14P11E201\AI\kyr'
os.chdir(base_path)
print(f"작업 디렉토리: {os.getcwd()}")

# MLflow 환경변수 초기화 (캐시 문제 해결)
if 'MLFLOW_TRACKING_URI' in os.environ:
    del os.environ['MLFLOW_TRACKING_URI']

# MLflow 설정 (Windows 호환)
mlruns_path = os.path.join(base_path, 'mlruns')
os.makedirs(mlruns_path, exist_ok=True)

# 기존 활성 run 종료
try:
    if mlflow.active_run():
        mlflow.end_run()
except:
    pass

# tracking URI 설정 (Windows: file:/// 형식 사용)
tracking_uri = f"file:///{mlruns_path.replace(os.sep, '/')}"
mlflow.set_tracking_uri(tracking_uri)

# 실험 이름 설정
experiment_name = "SafeDeck_YOLOv11_Detection"
mlflow.set_experiment(experiment_name)

print(f"MLflow Tracking URI: {tracking_uri}")
print(f"MLflow Experiment: {experiment_name}")
print("MLflow 설정 완료!")

작업 디렉토리: C:\Users\SSAFY\Desktop\S14P11E201\AI\kyr
MLflow Tracking URI: file:///C:/Users/SSAFY/Desktop/S14P11E201/AI/kyr/mlruns
MLflow Experiment: SafeDeck_YOLOv11_Detection
MLflow 설정 완료!


c:\Users\SSAFY\miniforge3\envs\safedeck\lib\site-packages\mlflow\tracking\_tracking_service\utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)


In [33]:
# ============================================================
# 학습 전략 (Training Strategy)
# ============================================================

training_strategy = """
┌─────────────────────────────────────────────────────────────┐
│                    학습 전략 요약                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  [1. 모델 아키텍처]                                          │
│      • 모델: YOLOv11 nano (yolo11n.pt)                      │
│      • 태스크: Object Detection (선박 도장 결함 탐지)         │
│      • 입력 크기: 640x640                                    │
│                                                             │
│  [2. 하이퍼파라미터]                                         │
│      • epochs: 100 (최대)                                   │
│      • batch: 16                                            │
│      • lr0: 0.02 (초기 학습률)                               │
│      • optimizer: AdamW                                     │
│      • patience: 5 (YOLO 기본 Early Stopping)               │
│                                                             │
│  [3. Smart Training Monitor (핵심)]                         │
│                                                             │
│      Train/Val Loss 실시간 분석으로 학습 상태 자동 판단      │
│                                                             │
│      ┌─────────────┬─────────────┬──────────────────┐       │
│      │ 상태        │ 조건        │ 액션             │       │
│      ├─────────────┼─────────────┼──────────────────┤       │
│      │ GOOD_FIT    │ Train↓ Val↓│ 계속 학습        │       │
│      │ OVERFITTING │ Train↓ Val↑│ 즉시 중단 (3회)  │       │
│      │ STUCK       │ Train→ Val→│ 중단 (5 에포크)  │       │
│      │ UNSTABLE    │ 진동        │ 경고 표시        │       │
│      └─────────────┴─────────────┴──────────────────┘       │
│                                                             │
│      보수적 설정 (시간 낭비 최소화):                         │
│      • overfitting_patience: 3 (3회 연속 Val↑ 시 중단)      │
│      • stuck_patience: 5 (5 에포크 개선 없으면 중단)        │
│      • min_delta: 0.001 (최소 개선 기준)                    │
│                                                             │
│  [4. 실시간 시각화]                                          │
│                                                             │
│      매 에포크마다 Jupyter 노트북에서 실시간 업데이트:       │
│      • Train vs Val Loss 곡선                               │
│      • Generalization Gap (Val - Train)                     │
│      • mAP50 성능 추이                                      │
│      • 현재 학습 상태 (색상 표시)                            │
│                                                             │
│  [5. 학습 흐름]                                              │
│                                                             │
│      학습 시작                                               │
│           │                                                 │
│           ▼                                                 │
│      ┌─────────────────────────────────┐                   │
│      │ [매 에포크]                      │                   │
│      │  1. 1 에포크 학습               │                   │
│      │  2. Loss 수집 (Train/Val)       │                   │
│      │  3. Smart Monitor 분석          │                   │
│      │  4. 실시간 그래프 업데이트       │                   │
│      │  5. 상태 판단                   │                   │
│      └─────────────────────────────────┘                   │
│           │                                                 │
│           ▼                                                 │
│      ┌─────────────────────────────────┐                   │
│      │ [상태 체크]                      │                   │
│      │  • GOOD_FIT → 계속              │                   │
│      │  • OVERFITTING → 즉시 중단      │                   │
│      │  • STUCK → 중단                 │                   │
│      └─────────────────────────────────┘                   │
│           │                                                 │
│           ▼                                                 │
│      best.pt = Best Val Loss 시점 모델                      │
│                                                             │
│  [6. 배포 파이프라인]                                        │
│                                                             │
│      PyTorch (.pt)                                          │
│           │                                                 │
│           ├──→ ONNX (.onnx) ──→ 범용 배포                   │
│           │                                                 │
│           └──→ TensorRT (.engine) ──→ Jetson 최적화 (FP16) │
│                                                             │
└─────────────────────────────────────────────────────────────┘
"""

print(training_strategy)


┌─────────────────────────────────────────────────────────────┐
│                    학습 전략 요약                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  [1. 모델 아키텍처]                                          │
│      • 모델: YOLOv11 nano (yolo11n.pt)                      │
│      • 태스크: Object Detection (선박 도장 결함 탐지)         │
│      • 입력 크기: 640x640                                    │
│                                                             │
│  [2. 하이퍼파라미터]                                         │
│      • epochs: 100 (최대)                                   │
│      • batch: 16                                            │
│      • lr0: 0.02 (초기 학습률)                               │
│      • optimizer: AdamW                                     │
│      • patience: 5 (YOLO 기본 Early Stopping)               │
│                                                             │
│  [3. Smart 

In [34]:
# ============================================================
# MLOps 전략 (MLOps Strategy)
# ============================================================

mlops_strategy = """
┌─────────────────────────────────────────────────────────────┐
│                    MLOps 전략 요약                           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  [1. 실험 추적 (Experiment Tracking)]                       │
│                                                             │
│      mlflow.set_experiment("SafeDeck_YOLOv11_Detection")    │
│                    │                                        │
│                    ▼                                        │
│      with mlflow.start_run(run_name=...) as run:            │
│          │                                                  │
│          ├── mlflow.log_params(...)   # 파라미터 기록       │
│          ├── mlflow.log_metric(...)   # 메트릭 기록         │
│          ├── mlflow.log_artifact(...) # 아티팩트 저장       │
│          └── mlflow.set_tag(...)      # 태그 분류           │
│                                                             │
│  [2. 로깅 대상]                                              │
│      ┌────────────────┬─────────────────────────────────┐   │
│      │ 카테고리       │ 내용                            │   │
│      ├────────────────┼─────────────────────────────────┤   │
│      │ Parameters     │ epochs, batch, lr0, optimizer   │   │
│      │ System Info    │ GPU, CUDA, PyTorch 버전         │   │
│      │ Metrics        │ mAP50, Precision, Recall, FPS   │   │
│      │ Artifacts      │ .pt, .onnx, .engine, 학습곡선   │   │
│      │ Tags           │ project, task, stage            │   │
│      └────────────────┴─────────────────────────────────┘   │
│                                                             │
│  [3. 파이프라인 단계별 MLflow Run]                           │
│                                                             │
│      SafeDeck_YOLOv11_Detection (Experiment)                │
│      │                                                      │
│      ├── [Run 1] YOLOv11_20260120_143000                   │
│      │      • stage: training                               │
│      │      • params: epochs, batch, lr0...                │
│      │      • metrics: mAP50, precision, recall            │
│      │      • artifacts: best.pt, results.csv              │
│      │                                                      │
│      ├── [Run 2] Validation_20260120_150000                │
│      │      • stage: validation                             │
│      │      • metrics: val_mAP50, val_mAP50-95             │
│      │                                                      │
│      ├── [Run 3] Export_TensorRT_20260120_151000           │
│      │      • stage: export                                 │
│      │      • artifacts: best.engine                        │
│      │                                                      │
│      ├── [Run 4] Export_ONNX_20260120_151500               │
│      │      • stage: export                                 │
│      │      • artifacts: best.onnx                          │
│      │                                                      │
│      └── [Run 5] Benchmark_20260120_152000                 │
│             • stage: benchmark                              │
│             • metrics: pytorch_fps, tensorrt_fps           │
│                                                             │
│  [4. 태그 기반 분류 체계]                                    │
│      ┌────────────────────┬──────────────────────────┐     │
│      │ Tag                │ 값                       │     │
│      ├────────────────────┼──────────────────────────┤     │
│      │ project            │ SafeDeck                 │     │
│      │ task               │ object_detection         │     │
│      │ model_architecture │ YOLOv11                  │     │
│      │ stage              │ training/validation/...  │     │
│      │ export_format      │ tensorrt/onnx            │     │
│      └────────────────────┴──────────────────────────┘     │
│                                                             │
│  [5. 아티팩트 저장 구조]                                     │
│                                                             │
│      mlruns/                                                │
│      └── {experiment_id}/                                   │
│          └── {run_id}/                                      │
│              ├── params/         # 하이퍼파라미터           │
│              ├── metrics/        # 성능 지표               │
│              ├── tags/           # 메타데이터              │
│              └── artifacts/                                 │
│                  ├── models/                                │
│                  │   ├── best.pt                            │
│                  │   ├── tensorrt/best.engine               │
│                  │   └── onnx/best.onnx                     │
│                  ├── metrics/results.csv                    │
│                  └── plots/*.png                            │
│                                                             │
│  [6. MLflow UI 대시보드]                                    │
│                                                             │
│      실행: mlflow ui --port 5000                            │
│      접속: http://localhost:5000                            │
│                                                             │
│      • Experiments: 모든 실험 목록                          │
│      • Runs 비교: 여러 학습 결과 병렬 비교                  │
│      • Metrics 시각화: mAP, Loss 차트                       │
│      • Artifacts 다운로드: 모델/파일 다운로드               │
│                                                             │
└─────────────────────────────────────────────────────────────┘
"""

print(mlops_strategy)


┌─────────────────────────────────────────────────────────────┐
│                    MLOps 전략 요약                           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  [1. 실험 추적 (Experiment Tracking)]                       │
│                                                             │
│      mlflow.set_experiment("SafeDeck_YOLOv11_Detection")    │
│                    │                                        │
│                    ▼                                        │
│      with mlflow.start_run(run_name=...) as run:            │
│          │                                                  │
│          ├── mlflow.log_params(...)   # 파라미터 기록       │
│          ├── mlflow.log_metric(...)   # 메트릭 기록         │
│          ├── mlflow.log_artifact(...) # 아티팩트 저장       │
│          └── mlflow.set_tag(...)      # 태그 분류           │
│                                                             │
│  

In [35]:
# ============================================================
# Smart Training Callback (실시간 Loss 모니터링 + 자동 중단)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from ultralytics.engine.trainer import BaseTrainer
from enum import Enum

class TrainingState(Enum):
    GOOD_FIT = "GOOD_FIT"           # 정상 학습: Train↓ Val↓
    OVERFITTING = "OVERFITTING"     # 과적합: Train↓ Val↑
    STUCK = "STUCK"                 # 정체: Train→ Val→
    UNSTABLE = "UNSTABLE"           # 불안정: 진동

class SmartTrainingMonitor:
    """
    Train/Val Loss를 실시간 모니터링하여 학습 상태를 판단하고
    필요시 조기 중단하는 스마트 콜백
    
    보수적 설정 (시간 낭비 최소화):
    - overfitting_patience: 3 에포크 연속 Val Loss 증가 시 중단
    - stuck_patience: 5 에포크 동안 개선 없으면 중단
    - min_delta: 0.001 (이 이상 변화해야 개선으로 인정)
    """
    
    def __init__(self, 
                 overfitting_patience=3,  # Val Loss 연속 증가 허용 횟수
                 stuck_patience=5,        # Loss 정체 허용 에포크
                 min_delta=0.001,         # 최소 개선 기준
                 window_size=3):          # 추세 분석 윈도우
        
        self.overfitting_patience = overfitting_patience
        self.stuck_patience = stuck_patience
        self.min_delta = min_delta
        self.window_size = window_size
        
        # Loss 기록
        self.train_losses = []
        self.val_losses = []
        self.epochs = []
        self.mAP50_history = []
        
        # 상태 추적
        self.current_state = TrainingState.GOOD_FIT
        self.val_increase_count = 0
        self.stuck_count = 0
        self.best_val_loss = float('inf')
        self.best_epoch = 0
        
        # 중단 플래그
        self.should_stop = False
        self.stop_reason = ""
        
        # 시각화
        self.fig = None
        self.axes = None
        
    def setup_plot(self):
        """실시간 시각화 설정"""
        plt.ion()  # 인터랙티브 모드
        self.fig, self.axes = plt.subplots(2, 2, figsize=(14, 10))
        self.fig.suptitle('Smart Training Monitor', fontsize=14, fontweight='bold')
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        
    def update(self, epoch, train_loss, val_loss, mAP50=None):
        """매 에포크 호출: Loss 기록 및 상태 분석"""
        self.epochs.append(epoch)
        self.train_losses.append(train_loss)
        self.val_losses.append(val_loss)
        if mAP50 is not None:
            self.mAP50_history.append(mAP50)
        
        # 상태 분석
        self._analyze_state()
        
        # 실시간 시각화
        self._update_plot()
        
        return self.should_stop
    
    def _analyze_state(self):
        """Train/Val Loss 관계 분석하여 학습 상태 판단"""
        if len(self.val_losses) < 2:
            return
        
        current_val = self.val_losses[-1]
        prev_val = self.val_losses[-2]
        current_train = self.train_losses[-1]
        prev_train = self.train_losses[-2]
        
        # 1. Overfitting 체크: Val Loss 증가 + Train Loss 감소
        val_increased = current_val > prev_val + self.min_delta
        train_decreased = current_train < prev_train - self.min_delta
        
        if val_increased and train_decreased:
            self.val_increase_count += 1
            if self.val_increase_count >= self.overfitting_patience:
                self.current_state = TrainingState.OVERFITTING
                self.should_stop = True
                self.stop_reason = f"OVERFITTING: Val Loss가 {self.overfitting_patience}회 연속 증가"
                return
        else:
            self.val_increase_count = 0
        
        # 2. Best 모델 갱신 체크
        if current_val < self.best_val_loss - self.min_delta:
            self.best_val_loss = current_val
            self.best_epoch = self.epochs[-1]
            self.stuck_count = 0
        else:
            self.stuck_count += 1
        
        # 3. Stuck 체크: 둘 다 개선 없음
        if self.stuck_count >= self.stuck_patience:
            self.current_state = TrainingState.STUCK
            self.should_stop = True
            self.stop_reason = f"STUCK: {self.stuck_patience}회 동안 Val Loss 개선 없음"
            return
        
        # 4. Unstable 체크: 최근 윈도우에서 진동
        if len(self.val_losses) >= self.window_size:
            recent_val = self.val_losses[-self.window_size:]
            val_std = np.std(recent_val)
            val_mean = np.mean(recent_val)
            
            if val_std / (val_mean + 1e-8) > 0.1:  # 변동계수 > 10%
                self.current_state = TrainingState.UNSTABLE
                # Unstable은 경고만, 중단하지 않음
            else:
                self.current_state = TrainingState.GOOD_FIT
    
    def _update_plot(self):
        """실시간 그래프 업데이트"""
        if self.fig is None:
            self.setup_plot()
        
        clear_output(wait=True)
        
        for ax in self.axes.flat:
            ax.clear()
        
        epochs = self.epochs
        
        # 1. Train vs Val Loss
        ax1 = self.axes[0, 0]
        ax1.plot(epochs, self.train_losses, 'b-', label='Train Loss', linewidth=2)
        ax1.plot(epochs, self.val_losses, 'r-', label='Val Loss', linewidth=2)
        ax1.axhline(y=self.best_val_loss, color='g', linestyle='--', alpha=0.5, label=f'Best Val: {self.best_val_loss:.4f}')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.set_title('Train vs Validation Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # 2. Loss Gap (Val - Train)
        ax2 = self.axes[0, 1]
        if len(self.train_losses) > 0:
            gap = [v - t for t, v in zip(self.train_losses, self.val_losses)]
            colors = ['red' if g > 0 else 'green' for g in gap]
            ax2.bar(epochs, gap, color=colors, alpha=0.7)
            ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
            ax2.set_xlabel('Epoch')
            ax2.set_ylabel('Gap (Val - Train)')
            ax2.set_title('Generalization Gap')
            ax2.grid(True, alpha=0.3)
        
        # 3. mAP50 추이
        ax3 = self.axes[1, 0]
        if len(self.mAP50_history) > 0:
            ax3.plot(epochs[:len(self.mAP50_history)], self.mAP50_history, 'g-', linewidth=2, marker='o', markersize=4)
            ax3.set_xlabel('Epoch')
            ax3.set_ylabel('mAP50')
            ax3.set_title('mAP50 Performance')
            ax3.grid(True, alpha=0.3)
            if len(self.mAP50_history) > 0:
                ax3.axhline(y=max(self.mAP50_history), color='orange', linestyle='--', alpha=0.5, label=f'Best: {max(self.mAP50_history):.4f}')
                ax3.legend()
        
        # 4. 학습 상태 표시
        ax4 = self.axes[1, 1]
        ax4.axis('off')
        
        # 상태별 색상
        state_colors = {
            TrainingState.GOOD_FIT: '#27ae60',      # 녹색
            TrainingState.OVERFITTING: '#e74c3c',   # 빨강
            TrainingState.STUCK: '#f39c12',         # 주황
            TrainingState.UNSTABLE: '#9b59b6'       # 보라
        }
        
        state_text = f"""
        ┌────────────────────────────────────────┐
        │         TRAINING STATUS                │
        ├────────────────────────────────────────┤
        │                                        │
        │  Current Epoch: {epochs[-1] if epochs else 0:>20}  │
        │  Best Epoch: {self.best_epoch:>23}  │
        │                                        │
        │  Train Loss: {self.train_losses[-1] if self.train_losses else 0:>23.4f}  │
        │  Val Loss: {self.val_losses[-1] if self.val_losses else 0:>25.4f}  │
        │  Best Val Loss: {self.best_val_loss:>20.4f}  │
        │                                        │
        │  Status: {self.current_state.value:>27}  │
        │  Stuck Count: {self.stuck_count}/{self.stuck_patience:>21}  │
        │  Overfit Count: {self.val_increase_count}/{self.overfitting_patience:>19}  │
        │                                        │
        └────────────────────────────────────────┘
        """
        
        ax4.text(0.5, 0.5, state_text, transform=ax4.transAxes, 
                fontsize=11, verticalalignment='center', horizontalalignment='center',
                fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor=state_colors[self.current_state], alpha=0.3))
        
        if self.should_stop:
            ax4.text(0.5, 0.1, f"⚠️ {self.stop_reason}", transform=ax4.transAxes,
                    fontsize=12, ha='center', color='red', fontweight='bold')
        
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        display(self.fig)
    
    def get_summary(self):
        """학습 종료 후 요약"""
        return {
            "total_epochs": len(self.epochs),
            "best_epoch": self.best_epoch,
            "best_val_loss": self.best_val_loss,
            "final_state": self.current_state.value,
            "stop_reason": self.stop_reason if self.should_stop else "Normal completion",
            "best_mAP50": max(self.mAP50_history) if self.mAP50_history else None
        }

# 전역 모니터 인스턴스
smart_monitor = None

print("SmartTrainingMonitor 클래스 로드 완료!")
print("""
설정값 (보수적):
  - overfitting_patience: 3 (Val Loss 3회 연속 증가 시 중단)
  - stuck_patience: 5 (5 에포크 개선 없으면 중단)
  - min_delta: 0.001 (최소 개선 기준)
""")

SmartTrainingMonitor 클래스 로드 완료!

설정값 (보수적):
  - overfitting_patience: 3 (Val Loss 3회 연속 증가 시 중단)
  - stuck_patience: 5 (5 에포크 개선 없으면 중단)
  - min_delta: 0.001 (최소 개선 기준)



---
# Phase 1: YOLOv11 학습 + MLflow 실험 추적

## V6 학습 설정
- **데이터셋**: Scratch 데이터 증강 (9장 → 1,637장)
- **Train**: 1,647장 / **Val**: 420장
- **MLflow**: 학습 파라미터, 메트릭, 모델 아티팩트 자동 기록
- **Smart Monitor**: Train/Val Loss 실시간 분석으로 과적합/정체 자동 감지

In [36]:
# ============================================================
# 빠른 테스트 (1 에포크만 실행하여 환경 검증)
# ============================================================

from ultralytics import YOLO
import torch

print("="*50)
print("  환경 검증 테스트 (1 에포크)")
print("="*50)

# 1. GPU 확인
print(f"\n[1] CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"    GPU: {torch.cuda.get_device_name(0)}")

# 2. 모델 로드 테스트
print("\n[2] 모델 로드 테스트...")
try:
    test_model = YOLO('yolo11n.pt')
    print("    ✅ 모델 로드 성공")
except Exception as e:
    print(f"    ❌ 모델 로드 실패: {e}")
    raise

# 3. 데이터셋 경로 확인
print("\n[3] 데이터셋 확인...")
import os
train_path = r'C:\Users\SSAFY\Desktop\S14P11E201\AI\kyr\images\train'
val_path = r'C:\Users\SSAFY\Desktop\S14P11E201\AI\kyr\images\val'
print(f"    Train 경로 존재: {os.path.exists(train_path)}")
print(f"    Val 경로 존재: {os.path.exists(val_path)}")

# 4. 1 에포크 학습 테스트
print("\n[4] 1 에포크 학습 테스트...")
try:
    test_results = test_model.train(
        data='ship_data.yaml',
        epochs=1,
        imgsz=640,
        batch=16,
        device=0,
        name='_test_run',
        exist_ok=True,
        verbose=False,
    )
    print("    ✅ 학습 테스트 성공!")
    print(f"    mAP50: {test_results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
except Exception as e:
    print(f"    ❌ 학습 테스트 실패: {e}")
    raise

# 5. MLflow 테스트
print("\n[5] MLflow 테스트...")
try:
    with mlflow.start_run(run_name="test_run") as run:
        mlflow.log_param("test", "success")
        mlflow.log_metric("test_metric", 1.0)
    mlflow.end_run()
    print("    ✅ MLflow 테스트 성공!")
except Exception as e:
    print(f"    ❌ MLflow 테스트 실패: {e}")
    raise

print("\n" + "="*50)
print("  ✅ 모든 테스트 통과! 본 학습 진행 가능")
print("="*50)

# 테스트 모델 정리
del test_model

  환경 검증 테스트 (1 에포크)

[1] CUDA 사용 가능: True
    GPU: NVIDIA GeForce RTX 4070 Laptop GPU

[2] 모델 로드 테스트...
    ✅ 모델 로드 성공

[3] 데이터셋 확인...
    Train 경로 존재: True
    Val 경로 존재: True

[4] 1 에포크 학습 테스트...
New https://pypi.org/project/ultralytics/8.4.6 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.253  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=ship_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kob

In [37]:
from ultralytics import YOLO
from ultralytics.utils import callbacks
import mlflow
import torch
import pandas as pd

# 학습 파라미터 정의
train_params = {
    "model_name": "yolo11n",
    "data_yaml": "ship_data.yaml",
    "epochs": 100,
    "imgsz": 640,
    "batch": 16,
    "lr0": 0.02,
    "patience": 5,       # 보수적 설정
    "optimizer": "AdamW",
    "device": 0,
}

# Smart Monitor 설정 (보수적)
smart_monitor = SmartTrainingMonitor(
    overfitting_patience=3,
    stuck_patience=5,
    min_delta=0.001
)

# YOLO 콜백 함수 정의
def on_train_epoch_end(trainer):
    """매 에포크 종료 시 호출되는 콜백"""
    epoch = trainer.epoch + 1
    
    # metrics에서 Loss 가져오기
    train_loss = trainer.loss.item() if hasattr(trainer.loss, 'item') else float(trainer.loss)
    
    # validator에서 val loss 가져오기
    val_loss = trainer.metrics.get('val/box_loss', 0) + \
               trainer.metrics.get('val/cls_loss', 0) + \
               trainer.metrics.get('val/dfl_loss', 0)
    
    mAP50 = trainer.metrics.get('metrics/mAP50(B)', 0)
    
    # Smart Monitor 업데이트
    should_stop = smart_monitor.update(epoch, train_loss, val_loss, mAP50)
    
    # MLflow 로깅
    mlflow.log_metric("train_loss", train_loss, step=epoch)
    mlflow.log_metric("val_loss", val_loss, step=epoch)
    mlflow.log_metric("mAP50", mAP50, step=epoch)
    
    # 조기 중단
    if should_stop:
        print(f"\n{'='*50}")
        print(f"Smart Monitor: {smart_monitor.stop_reason}")
        print(f"{'='*50}")
        trainer.stop = True

# 실험 run 이름 생성
run_name = f"YOLOv11_Smart_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# MLflow 실험 시작
with mlflow.start_run(run_name=run_name) as run:
    print(f"MLflow Run ID: {run.info.run_id}")
    print(f"MLflow Run Name: {run_name}")
    print("="*50)
    
    # 학습 파라미터 로깅
    mlflow.log_params(train_params)
    mlflow.log_param("cuda_available", torch.cuda.is_available())
    if torch.cuda.is_available():
        mlflow.log_param("gpu_name", torch.cuda.get_device_name(0))
    
    # YOLOv11 모델 로드
    model = YOLO(f'{train_params["model_name"]}.pt')
    
    # 콜백 등록
    model.add_callback("on_train_epoch_end", on_train_epoch_end)
    
    # 학습 시작 (한 번에 전체 에포크)
    results = model.train(
        data=train_params["data_yaml"],
        epochs=train_params["epochs"],
        imgsz=train_params["imgsz"],
        batch=train_params["batch"],
        name='SafeDeck_V5_YOLOv11_Smart',
        device=train_params["device"],
        lr0=train_params["lr0"],
        patience=train_params["patience"],
        optimizer=train_params["optimizer"],
        plots=True,
        save=True,
        exist_ok=True,
    )
    
    # 학습 완료 후 요약
    summary = smart_monitor.get_summary()
    results_dir = os.path.join(base_path, 'runs', 'detect', 'SafeDeck_V5_YOLOv11_Smart')
    best_model_path = os.path.join(results_dir, 'weights', 'best.pt')
    
    # MLflow 메트릭 로깅
    mlflow.log_metric("final_epoch", summary["total_epochs"])
    mlflow.log_metric("best_epoch", summary["best_epoch"])
    if summary["best_mAP50"]:
        mlflow.log_metric("best_mAP50", summary["best_mAP50"])
    
    mlflow.set_tag("final_state", summary["final_state"])
    mlflow.set_tag("project", "SafeDeck")
    mlflow.set_tag("training_mode", "smart_monitoring")
    
    # 모델 아티팩트 저장
    if os.path.exists(best_model_path):
        mlflow.log_artifact(best_model_path, artifact_path="models")
    
    print("\n" + "="*50)
    print("Smart Training 완료!")
    print("="*50)
    print(f"총 에포크: {summary['total_epochs']}")
    print(f"최고 성능 에포크: {summary['best_epoch']}")
    print(f"최종 상태: {summary['final_state']}")
    print(f"종료 이유: {summary['stop_reason']}")
    print("="*50)

<Figure size 1400x1000 with 4 Axes>

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 87/87 2.0it/s 43.2s0.5ss
                   all       2755       3591      0.694      0.759      0.783      0.515
EarlyStopping: Training stopped early as no improvement observed in last 5 epochs. Best results observed at epoch 38, best model saved as best.pt.
To update EarlyStopping(patience=5) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

43 epochs completed in 3.776 hours.
Optimizer stripped from C:\Users\SSAFY\Desktop\S14P11E201\AI\kyr\runs\detect\SafeDeck_V5_YOLOv11_Smart\weights\last.pt, 5.4MB
Optimizer stripped from C:\Users\SSAFY\Desktop\S14P11E201\AI\kyr\runs\detect\SafeDeck_V5_YOLOv11_Smart\weights\best.pt, 5.4MB

Validating C:\Users\SSAFY\Desktop\S14P11E201\AI\kyr\runs\detect\SafeDeck_V5_YOLOv11_Smart\weights\best.pt...
Ultralytics 8.3.253  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188M

In [3]:
# 기존 활성 run 종료
if mlflow.active_run():
    mlflow.end_run()

# 최고 성능 모델 로드 및 검증
best_model_path = os.path.join(base_path, 'runs', 'detect', 'SafeDeck_V5_YOLOv11_Smart', 'weights', 'best.pt')
model = YOLO(best_model_path)

metrics = model.val()

# 검증 결과를 MLflow에 추가 로깅
with mlflow.start_run(run_name=f"Validation_{datetime.now().strftime('%Y%m%d_%H%M%S')}") as run:
    mlflow.log_metric("val_mAP50", float(metrics.box.map50))
    mlflow.log_metric("val_mAP50-95", float(metrics.box.map))
    mlflow.set_tag("stage", "validation")
    mlflow.set_tag("model_path", best_model_path)

print(f"\n{'='*50}")
print(f"Phase 1 완료: YOLOv11 Smart Training 결과")
print(f"{'='*50}")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"{'='*50}")

NameError: name 'YOLO' is not defined

---
## Loss 모니터링 및 학습 곡선 분석

학습 중 기록된 Loss와 메트릭을 시각화하여 과적합 여부를 판단합니다.
- **Smart Training Monitor**: 실시간 Train/Val Loss 비교
- **자동 조기 종료**: OVERFITTING, STUCK 감지 시 학습 중단
- **MLflow**: 모든 메트릭 자동 기록

In [41]:
import pandas as pd
import matplotlib.pyplot as plt

# 학습 결과 로드
results_path = os.path.join(base_path, 'runs', 'detect', 'SafeDeck_V5_YOLOv11_Smart', 'results.csv')
df = pd.read_csv(results_path)
df.columns = df.columns.str.strip()  # 공백 제거

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Train Loss
axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='Box Loss', color='#e74c3c')
axes[0, 0].plot(df['epoch'], df['train/cls_loss'], label='Cls Loss', color='#3498db')
axes[0, 0].plot(df['epoch'], df['train/dfl_loss'], label='DFL Loss', color='#2ecc71')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Train Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Validation Loss
axes[0, 1].plot(df['epoch'], df['val/box_loss'], label='Box Loss', color='#e74c3c')
axes[0, 1].plot(df['epoch'], df['val/cls_loss'], label='Cls Loss', color='#3498db')
axes[0, 1].plot(df['epoch'], df['val/dfl_loss'], label='DFL Loss', color='#2ecc71')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_title('Validation Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. mAP 추이
axes[1, 0].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50', linewidth=2, color='#9b59b6')
axes[1, 0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95', linewidth=2, color='#f39c12')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('mAP')
axes[1, 0].set_title('mAP Performance')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Precision / Recall
axes[1, 1].plot(df['epoch'], df['metrics/precision(B)'], label='Precision', linewidth=2, color='#1abc9c')
axes[1, 1].plot(df['epoch'], df['metrics/recall(B)'], label='Recall', linewidth=2, color='#e67e22')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_title('Precision & Recall')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
training_curves_path = os.path.join(base_path, 'runs', 'detect', 'SafeDeck_V5_YOLOv11_Smart', 'training_curves.png')
plt.savefig(training_curves_path, dpi=150)
plt.show()

# 학습 곡선도 MLflow에 로깅
with mlflow.start_run(run_name=f"Analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}") as run:
    mlflow.log_artifact(training_curves_path, artifact_path="analysis")
    mlflow.set_tag("stage", "analysis")

# Best 에포크 분석
best_idx = df['metrics/mAP50(B)'].idxmax()
best_epoch = int(df.loc[best_idx, 'epoch'])
total_epochs = int(df['epoch'].max())

print(f"\n{'='*50}")
print(f"  학습 분석 결과")
print(f"{'='*50}")
print(f"총 학습 에포크: {total_epochs}")
print(f"최고 성능 에포크: {best_epoch}")
print(f"mAP50: {df['metrics/mAP50(B)'].max():.4f}")
print(f"mAP50-95: {df.loc[best_idx, 'metrics/mAP50-95(B)']:.4f}")
print(f"Precision: {df.loc[best_idx, 'metrics/precision(B)']:.4f}")
print(f"Recall: {df.loc[best_idx, 'metrics/recall(B)']:.4f}")
print(f"{'='*50}")

<Figure size 1400x1000 with 4 Axes>


  학습 분석 결과
총 학습 에포크: 43
최고 성능 에포크: 40
mAP50: 0.7995
mAP50-95: 0.5713
Precision: 0.7619
Recall: 0.7558


---
# Phase 2-A: ONNX 변환 (Windows 개발 환경)

## 역할: 범용 포맷 변환 및 검증

### ONNX가 필요한 이유
1. **범용성**: Windows/Linux/Jetson 모두 호환
2. **백업**: TensorRT 실패 시 ONNX Runtime 사용
3. **디버깅**: Netron으로 모델 구조 시각화

### 변환 파이프라인
```
Phase 1 (학습)  →  Phase 2-A (Windows)  →  Phase 2-B (Jetson)
   .pt          →       .onnx            →      .engine
  (학습)        →   (범용 포맷/검증)      →   (GPU 최적화)
```

In [1]:
# ============================================================
# ONNX 변환 (Windows 개발 환경)
# ============================================================

import platform

print("="*70)
print("  Phase 2-A: ONNX 변환")
print("="*70)

current_platform = platform.system()
print(f"\n현재 환경: {current_platform}")
print(f"변환 포맷: ONNX (범용)")

try:
    onnx_path = model.export(
        format='onnx',
        imgsz=640,
        simplify=True,
        opset=12
    )
    
    print(f"\n✅ ONNX 변환 성공!")
    print(f"   파일: {onnx_path}")
    
    # MLflow에 ONNX 모델 저장
    with mlflow.start_run(run_name=f"Export_ONNX_{datetime.now().strftime('%Y%m%d_%H%M%S')}") as run:
        mlflow.log_artifact(onnx_path, artifact_path="models/onnx")
        mlflow.set_tag("stage", "export")
        mlflow.set_tag("export_format", "onnx")
        mlflow.set_tag("platform", current_platform)
        mlflow.log_param("opset_version", 12)
        
    print(f"   MLflow 저장 완료")
    
except Exception as e:
    print(f"\n❌ ONNX 변환 실패: {e}")

print("\n" + "="*70)
print("  Phase 2-A 완료 - 다음: Phase 2-B (Jetson)")
print("="*70)

if current_platform == "Windows":
    print("\n⚠️  Windows 환경에서는 TensorRT 변환 불가")
    print("\n📌 Jetson Orin Nano에서 TensorRT 변환 방법:")
    print("\n" + "-"*70)
    print("방법 1: .pt 파일에서 직접 변환 (권장 - 간단함)")
    print("-"*70)
    print("# best.pt 파일을 Jetson으로 복사 후:")
    print("from ultralytics import YOLO")
    print("model = YOLO('best.pt')")
    print("model.export(format='engine', half=True, device=0)")
    print("# → best.engine 파일 생성 (FP16 최적화)")
    
    print("\n" + "-"*70)
    print("방법 2: ONNX 파일 경유 (권장 - 안정적)")
    print("-"*70)
    print("# best.onnx 파일을 Jetson으로 복사 후:")
    print("model = YOLO('best.onnx')")
    print("model.export(format='engine', half=True, device=0)")
    print("# 또는 trtexec 명령 사용:")
    print("/usr/src/tensorrt/bin/trtexec \\")
    print("  --onnx=best.onnx \\")
    print("  --saveEngine=best.engine \\")
    print("  --fp16 \\")
    print("  --workspace=4096")
    
    print("\n" + "-"*70)
    print("두 방법의 차이점")
    print("-"*70)
    print("방법 1 (.pt → TensorRT):")
    print("  ✅ 간단함 (한 단계)")
    print("  ✅ .pt 파일만 있으면 됨")
    print("  ❌ 내부적으로 ONNX 거침 (자동)")
    print("")
    print("방법 2 (ONNX → TensorRT):")
    print("  ✅ 디버깅 용이 (ONNX 시각화 가능)")
    print("  ✅ 세밀한 최적화 옵션 제어")
    print("  ✅ ONNX 백업본 보유")
    print("  ❌ 두 단계 필요")
    
    print("\n💡 권장: 방법 1로 시작, 문제 발생 시 방법 2 시도")
    
else:
    # Linux/Jetson 환경에서 TensorRT 변환 시도
    print("\nLinux 환경 감지: TensorRT 변환 시도...")
    try:
        import tensorrt as trt
        print(f"✅ TensorRT 버전: {trt.__version__}")
        
        print("\nTensorRT FP16 변환 시작...")
        tensorrt_path = model.export(
            format='engine',
            half=True,
            imgsz=640,
            device=0,
            simplify=True
        )
        
        print(f"\n✅ TensorRT 변환 성공!")
        print(f"   파일: {tensorrt_path}")
        
        # MLflow에 TensorRT 모델 저장
        with mlflow.start_run(run_name=f"Export_TensorRT_{datetime.now().strftime('%Y%m%d_%H%M%S')}") as run:
            mlflow.log_artifact(tensorrt_path, artifact_path="models/tensorrt")
            mlflow.set_tag("stage", "export")
            mlflow.set_tag("export_format", "tensorrt")
            mlflow.set_tag("platform", current_platform)
            mlflow.log_param("precision", "FP16")
            mlflow.log_param("device", 0)
            
        print(f"   MLflow에 저장 완료")
        
    except ImportError:
        print("\n⚠️  TensorRT가 설치되지 않았습니다.")
        print("   Jetson Orin Nano에서 변환하세요.")
    except Exception as e:
        print(f"\n❌ TensorRT 변환 실패: {e}")

print("\n" + "="*70)
print("  Phase 2 완료")
print("="*70)

TensorRT 엔진 변환 시작...


NameError: name 'model' is not defined

In [ ]:
# ============================================================
# 변환 결과 요약
# ============================================================

print("\n" + "="*70)
print("  변환 완료 요약")
print("="*70)

print("\n✅ 완료된 작업:")
print("   1. ✅ PyTorch 모델 (.pt) - Phase 1 학습 완료")
print("   2. ✅ ONNX 모델 (.onnx) - Phase 2 변환 완료")
print("   3. ⏭️  TFLite 모델 (.tflite) - 선택사항 (필요시 활성화)")
print("   4. ⏭️  TensorRT 엔진 (.engine) - Jetson에서 변환 예정")

print("\n📦 모델 파일 위치:")
print(f"   - PyTorch:  runs/detect/SafeDeck_V6_YOLOv11_Smart/weights/best.pt")
print(f"   - ONNX:     runs/detect/SafeDeck_V6_YOLOv11_Smart/weights/best.onnx")
print(f"   - MLflow:   mlruns/{mlflow.active_experiment().experiment_id}/")

print("\n📋 Jetson Orin Nano 배포 가이드:")
print("\n1️⃣  파일 전송 (둘 중 하나)")
print("   - best.pt → Jetson (직접 변환용)")
print("   - best.onnx → Jetson (ONNX 경유용)")

print("\n2️⃣  TensorRT 변환 (Jetson에서 실행)")
print("\n   [방법 A] .pt 파일 직접 변환 (간단)")
print("   ```python")
print("   from ultralytics import YOLO")
print("   model = YOLO('best.pt')")
print("   model.export(format='engine', half=True, device=0)")
print("   # 결과: best.engine 생성")
print("   ```")

print("\n   [방법 B] ONNX 파일 경유 (안정적)")
print("   ```bash")
print("   # trtexec로 변환 (더 세밀한 제어)")
print("   /usr/src/tensorrt/bin/trtexec \\")
print("     --onnx=best.onnx \\")
print("     --saveEngine=best.engine \\")
print("     --fp16 \\")
print("     --workspace=4096 \\")
print("     --minShapes=images:1x3x640x640 \\")
print("     --optShapes=images:1x3x640x640 \\")
print("     --maxShapes=images:1x3x640x640")
print("   ```")

print("\n3️⃣  추론 성능 테스트")
print("   ```python")
print("   import time")
print("   from ultralytics import YOLO")
print("   ")
print("   # TensorRT 모델 로드")
print("   model = YOLO('best.engine')")
print("   ")
print("   # FPS 측정")
print("   for i in range(100):")
print("       start = time.time()")
print("       results = model('test_image.jpg')")
print("       fps = 1 / (time.time() - start)")
print("       print(f'FPS: {fps:.1f}')")
print("   ```")

print("\n📊 예상 성능 비교 (Jetson Orin Nano):")
print("   ┌─────────────┬──────────┬────────────┬─────────┬─────────┐")
print("   │ 모델 포맷    │ FPS      │ 지연시간    │ 메모리   │ 추천도   │")
print("   ├─────────────┼──────────┼────────────┼─────────┼─────────┤")
print("   │ PyTorch     │ 20-30    │ ~33ms      │ 100%    │ ⭐      │")
print("   │ ONNX        │ 40-50    │ ~20ms      │ 80%     │ ⭐⭐    │")
print("   │ TensorRT    │ 60-90    │ ~11ms      │ 50%     │ ⭐⭐⭐   │")
print("   │ TFLite      │ 15-25    │ ~40ms      │ 70%     │ (CPU용) │")
print("   └─────────────┴──────────┴────────────┴─────────┴─────────┘")

print("\n💡 배포 권장사항:")
print("   1. 개발: PyTorch (.pt) - 디버깅 용이")
print("   2. 테스트: ONNX (.onnx) - 플랫폼 호환성 검증")
print("   3. 프로덕션: TensorRT (.engine) - 최대 성능")


---
# Phase 2-B: TensorRT 변환 (Jetson Orin Nano 전용)

⚠️ **이 섹션은 Jetson에서만 실행**

Jetson으로 `best.pt` 파일 복사 → TensorRT 변환 → 실시간 추론

In [ ]:
# ============================================================
# Jetson Orin Nano에서 TensorRT 변환
# ============================================================
# 이 셀은 Jetson 환경에서만 실행하세요!

import platform
import os
import torch
from ultralytics import YOLO

print("="*70)
print("  Phase 2-B: TensorRT 변환 (Jetson Orin Nano)")
print("="*70)

# 1. 환경 확인
current_platform = platform.system()
print(f"\n[1] 플랫폼 확인")
print(f"    현재 환경: {current_platform}")

if current_platform == "Windows":
    print("\n⚠️ 경고: Windows 환경에서는 이 셀을 실행하지 마세요!")
    print("   Jetson Orin Nano로 파일을 전송한 후 실행하세요.")
    raise SystemExit("Windows에서 TensorRT 변환 불가")

print(f"    ✅ Jetson/Linux 환경 확인")

# 2. CUDA 확인
print(f"\n[2] CUDA 확인")
print(f"    CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"    GPU: {torch.cuda.get_device_name(0)}")
    print(f"    CUDA 버전: {torch.version.cuda}")
    print(f"    ✅ GPU 사용 준비 완료")
else:
    print(f"    ⚠️ CUDA를 사용할 수 없습니다. CPU로 변환됩니다.")

# 3. 모델 파일 확인
print(f"\n[3] 모델 파일 확인")
model_path = 'runs/detect/SafeDeck_V6_YOLOv11_Smart/weights/best.pt'
if not os.path.exists(model_path):
    print(f"    ❌ 모델 파일을 찾을 수 없습니다: {model_path}")
    print(f"    best.pt 파일을 Jetson으로 복사했는지 확인하세요.")
    raise FileNotFoundError(model_path)

pt_size = os.path.getsize(model_path) / (1024 * 1024)
print(f"    파일: {model_path}")
print(f"    크기: {pt_size:.2f} MB")
print(f"    ✅ 모델 파일 확인 완료")

# 4. TensorRT 변환 시작
print("\n" + "="*70)
print("TensorRT FP16 변환 시작...")
print("="*70)
print("⏱️  예상 소요 시간: 2-5분")
print("📊 변환 설정:")
print("   - 정밀도: FP16 (속도 2배, 정확도 유지)")
print("   - 입력 크기: 640x640")
print("   - 배치 크기: 1 (실시간 추론용)")
print("\n💡 참고: Ultralytics가 자동으로 TensorRT를 사용합니다.")
print("   (TensorRT import 불필요 - 내부적으로 처리)")

try:
    model = YOLO(model_path)
    
    # TensorRT 변환 (TensorRT import 없이 작동!)
    engine_path = model.export(
        format='engine',      # TensorRT 엔진
        half=True,            # FP16 정밀도
        imgsz=640,            # 입력 이미지 크기
        device=0,             # GPU 0 사용
        simplify=True,        # ONNX 그래프 단순화
        workspace=4,          # 작업 공간 4GB
        verbose=True          # 상세 로그 출력
    )
    
    print(f"\n" + "="*70)
    print("✅ TensorRT 변환 성공!")
    print("="*70)
    print(f"📁 파일 위치: {engine_path}")
    
    # 파일 크기 비교
    engine_size = os.path.getsize(engine_path) / (1024 * 1024)
    
    print(f"\n📊 파일 크기 비교:")
    print(f"   - PyTorch (.pt):      {pt_size:.2f} MB")
    print(f"   - TensorRT (.engine): {engine_size:.2f} MB")
    print(f"   - 압축률: {(1 - engine_size/pt_size)*100:.1f}% 감소")
    
    print(f"\n💡 다음 단계:")
    print(f"   1. Phase 3: 추론 속도 벤치마크")
    print(f"   2. Flask API 서버 통합")
    print(f"   3. 실시간 카메라 테스트")
    
except Exception as e:
    print(f"\n❌ TensorRT 변환 실패!")
    print(f"   에러: {e}")
    print(f"\n💡 문제 해결:")
    print(f"   1. CUDA 메모리 부족: workspace 값을 2로 낮추기")
    print(f"      model.export(..., workspace=2)")
    print(f"   2. JetPack 버전 확인: sudo apt show nvidia-jetpack")
    print(f"   3. ONNX 경유 시도: 먼저 .onnx로 변환 후 .engine 생성")
    print(f"   4. TensorRT 로그 확인: verbose=True로 상세 로그 보기")
    raise

print("\n" + "="*70)
print("  Phase 2-B 완료!")
print("="*70)

---
# Phase 3: 속도 벤치마크 (PyTorch vs TensorRT)

V6 학습 완료 후, PyTorch와 TensorRT 모델의 추론 속도를 비교합니다.

In [ ]:
import time
import cv2
import numpy as np

# 테스트 이미지 준비
test_img_dir = os.path.join(base_path, 'images', 'val', 'Pinhole')
test_images = [os.path.join(test_img_dir, f) for f in os.listdir(test_img_dir)[:50]
               if f.lower().endswith(('.jpg', '.png'))]

def benchmark_model(model_path, name, num_runs=50):
    """모델 추론 속도 벤치마크"""
    model = YOLO(model_path)
    
    # Warmup
    for img_path in test_images[:5]:
        model.predict(img_path, verbose=False)
    
    # 벤치마크
    times = []
    for img_path in test_images[:num_runs]:
        start = time.time()
        model.predict(img_path, verbose=False)
        times.append(time.time() - start)
    
    avg_time = np.mean(times) * 1000  # ms
    fps = 1000 / avg_time
    
    print(f"{name}:")
    print(f"  평균 추론 시간: {avg_time:.1f}ms")
    print(f"  FPS: {fps:.1f}")
    
    return avg_time, fps

print("="*50)
print("  속도 벤치마크: PyTorch vs TensorRT")
print("="*50 + "\n")

# PyTorch 모델 벤치마크
pt_time, pt_fps = benchmark_model(best_model_path, "PyTorch (FP32)")

print()

# TensorRT 모델 벤치마크 (있으면 실행)
trt_time, trt_fps = None, None
if os.path.exists(tensorrt_model_path):
    trt_time, trt_fps = benchmark_model(tensorrt_model_path, "TensorRT (FP16)")
    
    print("\n" + "="*50)
    print(f"속도 향상: {pt_time/trt_time:.1f}x 빠름")
    print("="*50)
else:
    print("\n[TensorRT 엔진이 없습니다. Phase 2를 먼저 실행하세요.]")

# MLflow에 벤치마크 결과 로깅
with mlflow.start_run(run_name=f"Benchmark_{datetime.now().strftime('%Y%m%d_%H%M%S')}") as run:
    mlflow.log_metric("pytorch_inference_time_ms", pt_time)
    mlflow.log_metric("pytorch_fps", pt_fps)
    if trt_time and trt_fps:
        mlflow.log_metric("tensorrt_inference_time_ms", trt_time)
        mlflow.log_metric("tensorrt_fps", trt_fps)
        mlflow.log_metric("speedup_ratio", pt_time/trt_time)
    mlflow.set_tag("stage", "benchmark")

---
# Phase 4: 최종 결과 시각화

V6 학습 결과를 시각화하고, V5와 성능을 비교합니다.
- Scratch 클래스 mAP50 개선 여부 확인
- 전체 mAP50 목표(0.74+) 달성 여부 확인

In [ ]:
import matplotlib.pyplot as plt
import random

class_names = ['WaterSpotting', 'Sagging', 'Peeling', 'Pinhole', 'Crack',
               'Blistering', 'Inclusion', 'WeldingDamage', 'Scratch', 'Corrosion']

val_img_dir = os.path.join(base_path, 'images', 'val')
output_dir = os.path.join(base_path, 'outputs_V5')
os.makedirs(output_dir, exist_ok=True)

# 클래스별 폴더에서 이미지 수집
all_val_images = []
for class_name in class_names:
    class_dir = os.path.join(val_img_dir, class_name)
    if os.path.exists(class_dir):
        for f in os.listdir(class_dir):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_val_images.append(os.path.join(class_dir, f))

# 랜덤 10장 선택
sample_images = random.sample(all_val_images, min(10, len(all_val_images)))

# TensorRT 모델 사용 (없으면 PyTorch)
if os.path.exists(tensorrt_model_path):
    infer_model = YOLO(tensorrt_model_path)
    model_name = "YOLOv11 + TensorRT"
else:
    infer_model = YOLO(best_model_path)
    model_name = "YOLOv11 (PyTorch)"

for i, img_path in enumerate(sample_images):
    results = infer_model.predict(source=img_path, conf=0.25, device=0)[0]
    
    orig_img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    labeled_img = cv2.cvtColor(results.plot(), cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(20, 10))
    plt.subplot(1, 2, 1)
    plt.imshow(orig_img)
    plt.title(f"Original: {os.path.basename(img_path)}", fontsize=15)
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(labeled_img)
    plt.title(f"SafeDeck AI - {model_name}", fontsize=15)
    plt.axis('off')
    
    plt.tight_layout()
    save_path = os.path.join(output_dir, f"v5_result_{i+1}.jpg")
    plt.savefig(save_path)
    plt.close()
    print(f"[{i+1}/10] 저장: {save_path}")

---
# Phase 5: 최종 요약 - V6 목표 달성!

## V6 실험 결과

| 지표 | V5 | V6 | 목표 | 달성 |
|------|-----|-----|------|------|
| **mAP50** | 0.6995 | **0.7972** | 0.74+ | ✅ 초과 달성 |
| **Scratch mAP50** | 0.116 | **0.812** | 0.5+ | ✅ 초과 달성 |
| **mAP50-95** | 0.5033 | **0.5740** | - | ✅ 개선 |

## 핵심 성과
1. **Scratch 클래스**: mAP50 0.116 → **0.812** (7배 향상)
2. **전체 mAP50**: 0.6995 → **0.7972** (+9.8%p)
3. **데이터 증강**: AI Hub 데이터 1,637장 추가

## 다음 단계
- [x] Scratch 데이터 증강 (AI Hub)
- [x] V6 학습 및 목표 달성
- [ ] TensorRT 변환 및 속도 벤치마크
- [ ] Jetson Orin Nano 배포 테스트
- [ ] (선택) Pinhole/Peeling 추가 증강

In [ ]:
import os

print("\n" + "="*60)
print("  SafeDeck V5: YOLOv11 + MLflow + TensorRT 파이프라인 완료")
print("="*60)

print("\n[생성된 모델 파일]")
print(f"  PyTorch:   {best_model_path}")
if os.path.exists(best_model_path.replace('.pt', '.onnx')):
    print(f"  ONNX:      {best_model_path.replace('.pt', '.onnx')}")
if os.path.exists(best_model_path.replace('.pt', '.engine')):
    print(f"  TensorRT:  {best_model_path.replace('.pt', '.engine')}")

print("\n[성능 지표]")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")

print("\n[MLflow 실험 관리]")
print(f"  Tracking URI: {mlflow_tracking_uri}")
print(f"  Experiment: {experiment_name}")
print(f"  대시보드 실행: mlflow ui --backend-store-uri file://{mlflow_tracking_uri}")

print("\n[배포 옵션]")
print("  - Jetson Orin Nano: .engine (TensorRT FP16)")
print("  - 서버/클라우드: .pt (PyTorch) 또는 .onnx")
print("  - 엣지 디바이스: .onnx → TFLite 변환 가능")

print("\n" + "="*60)